# Tutorial 02: Simulating dynamical properties of neutron stars

We can perform the simulation of a Galactic population of neutron stars in different ways.

In this tutorial, we will focus on performing the dynamical evolution only and use the following command:
```
python mlpoppyns/simulator/simulate_population_dyn.py --save_dir output/sim_dyn
```
As for the end-to-end simualtion introduced in Tutorial 01, to change the initial parameters, the user can directly modify the simulator 
configuration in `mlpoppyns/simulator/config_simulator.py` or alternatively parse a JSON file containing custom simulation parameters.

The above command will create a population of neutron stars according to the initial conditions specified in `mlpoppyns/simulator/config_simulator.py` and evolve it in time dynamically. The output is saved in the specified output folder and consists of the following files:

* `final_pop_dyn.csv` containing the information on the final positions and velocities of neutron stars in the Galaxy.
* `.json` and `.log` files containing the timing profiles for the simulation, if enabled.
* `configuration.json` file containing the configuration parameters for reproducibility.

For an application of this simulation mode and the various physical models implemented see [Ronchi et al. (2021)](https://ui.adsabs.harvard.edu/abs/2021ApJ...916..100R/abstract).

In [ ]:
import argparse
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys

import utilities.plot_settings
from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.stellar_dynamics.coordinate_conversions as cc
from mlpoppyns.simulator.simulate_population_dyn import simulate_population

WARNING: If you see a warning here, make sure to set the path to the repository in the simulator configuration file. Otherwise, the examples below will not run.

## Setting up and running the dynamical simulation

We can adjust several parameters in the imported configuration file.
For example, we can change the following:
1. `NS_number`: number of neutron stars to simulate.
2. `t_age_max`: the maximum age for the simulated neutron stars in [yr].
3. `kick_model`: the kick-velocity model; we can choose either a Maxwell distribution, `km_maxwell` or an exponential distribution, `km_exp`.
4. `sigma_k`: the dispersion of the kick-velocity distribution, if `kick_model = km_maxwell`.
5. `vk_c`: the characteristic velocity of the kick-velocity distribution, if `kick_model = km_exp`.
6. `h_c`: the characteristic scale height of the Galactic exponential disk model.

NOTE: To obtain a realistic birth rate ($\sim 1$ neutron star per century) and a reasonable observed pulsar population, we need to set `NS_number` and `t_age_max` accordingly. However, by increasing both `NS_number` and `t_age_max` the simulation will take more time to run.

We will define the following simulation parameters:

In [ ]:
cfg["NS_number"] = 10000
cfg["t_age_max"] = 3.0e7
cfg["kick_model"] = "km_maxwell"
cfg["sigma_k"] = 265.0
cfg["vk_c"] = 180.0
cfg["h_c"] = 0.18

Alternatively, we can directly change the parameters in the `parameter_override.json` file in the `tutorials/tutorial_notebooks` folder and pass it to the simulator. Note that we will not use this in the example below, however.

In [ ]:
override_dir = "parameter_override.json"

We next specify the output directory where the simulation results will be saved.

In [ ]:
output_dir = "output/sim_dyn"

We can now run the simulation by calling the `simulate_population` function from the `mlpoppyns.simulator.simulate_population_dyn` module with our specific parameter choices.

In [ ]:
simulation_args = argparse.Namespace(
    save_dir=output_dir, parameter_override=None
)
simulate_population(simulation_args)

## Reading the simulation results

We first read in our compressed `.csv` population file.

In [ ]:
data_dyn = pd.read_csv(
    pathlib.Path().joinpath(output_dir, "final_pop_dyn.csv"),
    delimiter=",",
    header=[0, 1],
)
data_dyn.columns

Extracting the parameters of the final population.

In [ ]:
r = data_dyn["r"]["[kpc]"].to_numpy()
phi = data_dyn["phi"]["[rad]"].to_numpy()
z = data_dyn["z"]["[kpc]"].to_numpy()
v_r = data_dyn["v_r"]["[km/s]"].to_numpy()
v_phi = data_dyn["v_phi"]["[km/s]"].to_numpy()
v_z = data_dyn["v_z"]["[km/s]"].to_numpy()
age = data_dyn["age"]["[yr]"].to_numpy()

x = r * np.cos(phi)
y = r * np.sin(phi)

## Plotting the simulation results

We first look at the distribution of our final population in the Galactic plane.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=1,
    alpha=0.5,
    rasterized=True,
)
ax.plot(
    0.0,
    8.3,
    linestyle="None",
    marker="*",
    color="tab:orange",
    markersize=20,
    label="Sun",
)
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)
plt.xlabel(r"$x$ [kpc]")
plt.ylabel(r"$y$ [kpc]")
plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20)

plt.show()

From the side, we obtain the following:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=1,
    alpha=0.5,
    rasterized=True,
)

ax.plot(
    0.0,
    0.02,
    linestyle="None",
    marker="*",
    color="tab:orange",
    markersize=20,
    label="Sun",
)
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)
plt.xlabel(r"$x$ [kpc]")
plt.ylabel(r"$z$ [kpc]")
plt.legend(bbox_to_anchor=(1, 1), frameon=False, loc=0, fontsize=20)

plt.show()